In [ ]:
import pandas as pd
import numpy as np 
from datetime import timedelta
import os
import calendar
import pandas as pd
import warnings
import pyreadr
from pandas.tseries.offsets import MonthEnd
import matplotlib.pyplot as plt
import pyarrow.feather as feather
warnings.filterwarnings('ignore')

In [ ]:
def create_casecrossover_df(df_input):
    # Add a month_range column to df
    df_input['month_range'] = df_input.apply(lambda x: calendar.monthrange(x['year'], x['month'])[1], axis=1)
    # Calculate the weekday for the given year, month, day
    df_input['date'] = pd.to_datetime(df_input[['year', 'month', 'day']])
    df_input['weekday'] = df_input['date'].dt.dayofweek
    # Function to find matching days
    def find_matching_days(row):
        days_in_month = np.arange(1, row['month_range'] + 1)
        matching_days = days_in_month[np.array([(pd.Timestamp(row['year'], row['month'], day).dayofweek == row['weekday']) for day in days_in_month])]
        return matching_days[matching_days != row['day']]
    # Apply function to find all matching days
    df_input['matching_days'] = df_input.apply(find_matching_days, axis=1)
    # Initialize a list to hold the new rows
    new_rows = []
    # Iterate over the dataframe using itertuples for efficiency
    for row in df_input.itertuples():
        # Append the original row to the list as a dict
        new_rows.append(row._asdict())
        # Create the additional rows for matching days
        for day in row.matching_days:
            new_row = row._asdict()
            new_row['day'] = day
            new_row['patient'] = 0  # Set patient to 0 for the new rows
            new_rows.append(new_row)
    # Create a new dataframe from the list of dicts
    casecrossover_df = pd.DataFrame(new_rows)

    # Remove the index and other unnecessary columns added by _asdict()
    casecrossover_df.drop(columns=['Index', 'month_range', 'weekday', 'matching_days', 'date'], inplace=True)
    return casecrossover_df

def create_lagged_df(df_input):
    # Ensure 'date' column is in datetime format
    df_input['date'] = pd.to_datetime(df_input['date'])
    # Sort the dataframe by 'LocIDSmall' and 'date' to ensure the lags make chronological sense
    df_input = df_input.sort_values(by=['LocIDLarge', 'date'])
    # Initialize a dataframe to hold the lagged values
    lagged_df = pd.DataFrame()
    # Group the dataframe by 'LocIDSmall' to perform the operation within each location
    grouped = df_input.groupby('LocIDLarge')
    # Loop to create columns for each lag day up to 30 days
    for i in range(-4, 40):
        # Create lagged columns for 'tmean' and 'RH'
        df_input[f'tmean_lag{i}'] = grouped['tmean'].shift(i)
        df_input[f'RH_lag{i}'] = grouped['RH'].shift(i)
        df_input[f'tmean_temperature_lag{i}'] = grouped['tmean_temperature'].shift(i)
    # Keep only the columns of interest
    columns_of_interest = ['date', 'LocIDLarge'] + [f'tmean_lag{i}' for i in range(-4, 40)] + [f'RH_lag{i}' for i in range(0, 40)] + [f'tmean_temperature_lag{i}' for i in range(0, 40)]
    lagged_df = df_input[columns_of_interest]
    #add new columns to the lagged_df year month day and remove the date column
    lagged_df['year'] = lagged_df['date'].dt.year
    lagged_df['month'] = lagged_df['date'].dt.month
    lagged_df['day'] = lagged_df['date'].dt.day
    # lagged_df['tmean_temperture'] = df_input['tmean_temperature']
    lagged_df.drop(columns=['date'], inplace=True)
    return lagged_df


def calculate_hot_season(df):
    # Group by LocIDLarge, year, and month and calculate the mean of tmean
    grouped = df.groupby(['LocIDLarge', 'year', 'month']).agg({'tmean': 'mean'}).reset_index()
    # Calculate rolling mean over 4 months within each group of LocIDLarge
    grouped['tmean_mean_4month'] = grouped.groupby('LocIDLarge')['tmean'].transform(lambda x: x.rolling(4).mean())
    # Find the month with the max tmean_mean_4month for each LocIDLarge
    max_months = grouped.loc[grouped.groupby('LocIDLarge')['tmean_mean_4month'].idxmax()]
    # Define a function to calculate hot months
    def get_hot_months(row):
        base_month = row['month']
        return [((base_month - i - 1) % 12) + 1 for i in range(4)]
    # Apply the function
    max_months['hot_months'] = max_months.apply(get_hot_months, axis=1)
    # Create a dataframe for all months
    all_months = pd.DataFrame({'month': list(range(1, 13))})
    # Merge and create the hot_season flag
    df_weather_with_hotseason = pd.merge(max_months[['LocIDLarge', 'hot_months']], all_months, how='cross')
    df_weather_with_hotseason['hot_season'] = df_weather_with_hotseason.apply(lambda x: 1 if x['month'] in x['hot_months'] else 0, axis=1)
    # Drop the hot_months column
    df_weather_with_hotseason = df_weather_with_hotseason.drop(columns=['hot_months'])
    
    return df_weather_with_hotseason

In [ ]:
root_path = "XXX"
country_list = [ 'Brazil']
country_iso_lookup = { 'BRA' : 'Brazil', 'CAN' : 'Canada', 'CHL' : 'Chile', 'NZL' : 'New_Zealand'}
country_iso_list = ['BRA', 'CAN', 'CHL', 'NZL']
root_weather = "/Weather_data"
file_list_weather = ['MCC_LocIDLarge_AUS_2016_daily_weather_2000_2021.rds', 'MCC_LocIDLarge_countries_withoutAUS_daily_weather_2000_2021.rds']
data_type = ['Hospitalization']
outcome = ["F"]
look_up_table ={
    'DEM': ['F00', 'F01', 'F02', 'F03'],
    'SUB': ['F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19'],
    'SCHZ': ['F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29'],
    'MAN': ['F30'],
    'BPAD': ['F31'],
    'DEP': ['F32', 'F33'],
    'PMD': ['F34'],
    'OMD': ['F38'],
    'UMD': ['F39'],
    'ANX': ['F40', 'F41'],
    'OCD': ['F42'],
    'RSAD': ['F43'],
    'DCD': ['F44'],
    'SFD': ['F45'],
    'OND': ['F48'],
    'PD': ['F60', 'F61', 'F62', 'F63', 'F64', 'F65', 'F66', 'F68', 'F69'],
    'MR': ['F70', 'F71', 'F72', 'F73', 'F78', 'F79'],
    'DPD': ['F80', 'F81', 'F82', 'F83', 'F84', 'F88', 'F89'],
    'BED': ['F90', 'F91', 'F92', 'F93', 'F94', 'F95', 'F98'],
    'UMD': ['F99']
}

disease_lookup = {
    'DEM': 'Dementia',
    'SUB': 'Substance use',
    'SCHZ': 'Schizophrenia',
    'MAN': 'Manic episode',
    'BPAD': 'Bipolar affective disorder',
    'DEP': 'Depression',
    'PMD': 'Persistent mood [affective] disorders',
    'OMD': 'Other mood [affective] disorders',
    'UMD': 'Unspecified mood [affective] disorder',
    'ANX': 'Anxiety',
    'OCD': 'Obsessive-compulsive disorder',
    'RSAD': 'Reaction to severe stress, and adjustment disorders',
    'DCD': 'Dissociative [conversion] disorders',
    'SFD': 'Somatoform disorders',
    'OND': 'Other neurotic disorders',
    'PD': 'Personality disorders',
    'MR': 'Mental retardation',
    'DPD': 'Disorders of psychological development',
    'BED': 'Behavioral and emotional disorders with onset usually occurring in childhood and adolescence',
    'UMD': 'Unspecified mental disorder'
}


sum_df_list = []
sum_df = pd.DataFrame()
for country in country_list:
    for data in data_type:
        for out in outcome:
            file_name = data + "_" + out + "_" + country
            file_list = [os.path.join(root_path, f) for f in os.listdir(root_path) if f.startswith(file_name)] 
            for file in file_list:
                df_tem = pd.read_csv(file)
                df_tem['country'] = country
                sum_df_list.append(df_tem)
sum_df = pd.concat(sum_df_list, axis=0, ignore_index=True)
sum_df['disease'] = np.nan
for key, value in look_up_table.items():
    sum_df.loc[sum_df['icd10_3d'].str.startswith(tuple(value)), 'disease'] = key
sum_df = sum_df[sum_df['disease'].notna()]
sum_df['patient'] = 1
sum_df.drop(columns=['LocIDSmall'], inplace=True)
sum_brazil = sum_df[sum_df['country'] == 'Brazil']
sum_brazil = sum_brazil[['LocIDLarge', 'year', 'month', 'day']]
sum_df_counts = sum_brazil.groupby(['LocIDLarge', 'year', 'month', 'day']).size().reset_index(name='counts')
sum_df_counts['99th_percentile'] = sum_df_counts.groupby('LocIDLarge')['counts'].transform(lambda x: x.quantile(0.99))
sum_df_counts['99th_percentile'] = sum_df_counts.groupby('LocIDLarge')['counts'].transform(lambda x: x.quantile(0.99))
sum_df_counts['above_99th_percentile'] = sum_df_counts['counts'] > sum_df_counts['99th_percentile']
sum_df_counts.drop(columns=['counts', '99th_percentile'], inplace=True)
sum_df = sum_df.merge(sum_df_counts, on=['LocIDLarge', 'year', 'month', 'day'], how='left')
#remove rows with counts above 99th percentile true
sum_df = sum_df[sum_df['above_99th_percentile'] != True]
sum_df.drop(columns=['above_99th_percentile'], inplace=True)
LocIDLarge_list_withpatient = sum_df['LocIDLarge'].unique().tolist()

In [ ]:
root_holiday = "country_official_holidays_1986_2023.rds"
df_h = pyreadr.read_r(root_holiday)[None]
#use iso to filter the holiday data
df_h = df_h[df_h['ISO'].isin(country_iso_list)]
#add new column country based on the country_iso_lookup
df_h['country'] = df_h['ISO'].map(country_iso_lookup)
df_h['holiday_date'] = pd.to_datetime(df_h['holiday_date'])
df_h['year'] = pd.DatetimeIndex(df_h['holiday_date']).year
df_h['month'] = pd.DatetimeIndex(df_h['holiday_date']).month
df_h['day'] = pd.DatetimeIndex(df_h['holiday_date']).day
df_h['holiday'] = 1
df_h = df_h[['year','month','day','country','holiday']]
df = pd.DataFrame()
df_weather_list = []
for file in file_list_weather:
    df_tem = pyreadr.read_r(os.path.join(root_weather, file))[None]
    df_weather_list.append(df_tem)
df = pd.concat(df_weather_list, axis=0, ignore_index=True)
df = df[df['LocIDLarge'].isin(LocIDLarge_list_withpatient)]
df = df[['date', 'LocIDLarge', 'tmean', 'RH']]
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
#change the column name tmean to tmean_temperature
df = df.rename(columns={'tmean': 'tmean_temperature'})
#add new column tmean as the percentiles of the tmean_temperature based on each LocIDLarge
df['tmean'] = df.groupby('LocIDLarge')['tmean_temperature'].rank(pct=True)*100
df_weather_with_hotseason = calculate_hot_season(df)
sum_df = pd.merge(sum_df, df_weather_with_hotseason, how='left', on=['LocIDLarge', 'month'])
sum_df = sum_df[sum_df['hot_season'] == 1]
sum_df['id'] = np.arange(1, len(sum_df)+1)
case_crossover_df = create_casecrossover_df(sum_df)
df_weather_lagged = create_lagged_df(df)
case_crossover_df = case_crossover_df.merge(df_weather_lagged, how='left', on=['year', 'month', 'day', 'LocIDLarge'])
case_crossover_df = case_crossover_df.merge(df_h, how='left', on=['year','month','day','country'])
case_crossover_df['holiday'] = case_crossover_df['holiday'].fillna(0)
case_crossover_df = case_crossover_df.dropna()
feather.write_feather(case_crossover_df, 'case_crossover_df.feather')